# 01 — Exploration de CATMuS Medieval

**Objectif** : savoir quelles langues contient [CATMuS/medieval](https://huggingface.co/datasets/CATMuS/medieval) et combien de lignes en occitan il propose.

**Résultat** : 349 lignes d'occitan seulement, toutes issues d'un seul manuscrit (BnF fr. 13509, projet HTRogène). D'où le choix de travailler directement sur le dépôt HTRogène (notebook 02).

**Environnement** : Google Colab, sans GPU. Nécessite un token Hugging Face (lecture), enregistré dans les Secrets Colab sous le nom `HF_TOKEN`.

In [ ]:
!pip install -q -U datasets duckdb

import io, time
from collections import Counter
from getpass import getpass
import duckdb
from PIL import Image
from IPython.display import display
from datasets import load_dataset
from huggingface_hub import login

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = getpass("Token Hugging Face : ")
login(token=HF_TOKEN)

## 1. Structure d'un exemple

Lecture en streaming : on ne télécharge pas les 22,5 Go du jeu.

In [ ]:
ds = load_dataset("CATMuS/medieval", split="train", streaming=True)
ex = next(iter(ds))
for cle, valeur in ex.items():
    print(f"{cle:12} -> {('image ' + str(valeur.size)) if cle == 'im' else valeur}")

- Colonnes : `text`, `im`, `language`, `century`, `region`, `script_type`, `shelfmark`, `verse`, `genre`, `project`, `line_type`, `gen_split`.
- Les lignes sont **rangées par manuscrit** : parcourir le jeu en streaming pour trouver une langue rare est beaucoup trop lent.
- `gen_split` donne le découpage officiel (90/5/5, par manuscrit), indépendant du split Hugging Face.

## 2. Compter les langues avec DuckDB

DuckDB lit seulement la colonne `language` des fichiers Parquet, sans les images. Lecture fichier par fichier, avec pauses et nouvelles tentatives, pour éviter les erreurs HTTP 429 (limite de requêtes du Hub). La cellule peut être relancée : elle reprend là où elle s'est arrêtée.

In [ ]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}');")
con.sql("SET threads = 1;")
con.sql("SET http_retries = 10; SET http_retry_wait_ms = 3000; SET http_retry_backoff = 2;")

BASE = "hf://datasets/CATMuS/medieval@~parquet/default/train"
if "deja_lus" not in globals():
    deja_lus, compte = set(), Counter()

for i in range(252):
    if i in deja_lus:
        continue
    for langue, n in con.sql(f"SELECT language, COUNT(*) FROM '{BASE}/{i:04d}.parquet' GROUP BY language").fetchall():
        compte[langue] += n
    deja_lus.add(i)
    time.sleep(1)

for langue, n in compte.most_common():
    print(f"{langue:15} {n:6}")

## 3. Les lignes en occitan

In [ ]:
URL = f"{BASE}/*.parquet"
con.sql(f'''
    SELECT project, shelfmark, gen_split, COUNT(*) AS n
    FROM '{URL}' WHERE language = 'Occitan'
    GROUP BY ALL ORDER BY n DESC
''').show()

for img_bytes, texte in con.sql(f"SELECT im.bytes, text FROM '{URL}' WHERE language = 'Occitan' LIMIT 5").fetchall():
    display(Image.open(io.BytesIO(img_bytes)))
    print(repr(texte))